# Laboratório — Bernoulli, binomial, categorical e multinomial

Vamos construir, simular e validar quatro distribuições discretas usadas em IA.

**Dependências:** Python 3.10+, NumPy 1.24+, SciPy 1.10+ e Matplotlib 3.7+.  
**Reprodutibilidade:** seed `20260907`; tolerâncias declaradas antes das simulações.


## 1. Preparação

In [ ]:
import sys
import numpy as np
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

SEED = 20260907
TOL = 0.005
rng = np.random.default_rng(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")


## 2. Bernoulli

Codificamos clique como 1 e não clique como 0, com $p=0{,}17$.


In [ ]:
p_click = 0.17
suporte_bern = np.array([0, 1])
pmf_bern = stats.bernoulli.pmf(suporte_bern, p_click)
media_bern, var_bern = stats.bernoulli.stats(p_click, moments="mv")

assert np.isclose(pmf_bern.sum(), 1)
assert np.allclose(pmf_bern, [1-p_click, p_click])
assert np.isclose(media_bern, p_click)
assert np.isclose(var_bern, p_click * (1-p_click))
print("PMF [P(0), P(1)]:", pmf_bern)
print(f"E[X]={media_bern:.4f}; Var(X)={var_bern:.4f}")


In [ ]:
amostra_bern = rng.binomial(1, p_click, size=200_000)
assert abs(amostra_bern.mean() - p_click) < TOL
assert abs(amostra_bern.var() - var_bern) < TOL
print(f"Média teórica / simulada: {p_click:.6f} / {amostra_bern.mean():.6f}")
print(f"Variância teórica / simulada: {var_bern:.6f} / {amostra_bern.var():.6f}")


## 3. Binomial: PMF completa e caudas

Em dez exposições independentes com $p=0{,}2$, calculamos exatamente dois cliques e ao menos um clique.


In [ ]:
n, p = 10, 0.2
k = np.arange(n + 1)
pmf_bin = stats.binom.pmf(k, n, p)
p_exatamente_2 = stats.binom.pmf(2, n, p)
p_ao_menos_1 = stats.binom.sf(0, n, p)
media_bin, var_bin = stats.binom.stats(n, p, moments="mv")

assert np.isclose(pmf_bin.sum(), 1)
assert np.isclose(p_exatamente_2, 0.301989888)
assert np.isclose(p_ao_menos_1, 0.8926258176)
assert np.isclose(media_bin, 2.0) and np.isclose(var_bin, 1.6)
print(f"P(S=2) = {p_exatamente_2:.9f}")
print(f"P(S≥1) = {p_ao_menos_1:.10f}")
print(f"E[S]={media_bin:.2f}; Var(S)={var_bin:.2f}")


In [ ]:
campanhas = rng.binomial(n, p, size=200_000)
assert abs(campanhas.mean() - media_bin) < 0.02
assert abs(campanhas.var() - var_bin) < 0.03

freq_bin = np.bincount(campanhas, minlength=n+1) / len(campanhas)
fig, ax = plt.subplots(figsize=(8, 4))
ax.vlines(k, 0, pmf_bin, color="#2563eb", lw=3, label="PMF exata")
ax.scatter(k, pmf_bin, color="#2563eb")
ax.scatter(k, freq_bin, marker="x", s=50, color="#f59e0b", label="Frequência simulada")
ax.set(xlabel="Número de cliques S", ylabel="Probabilidade", title="Binomial(10, 0,2)", xticks=k)
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()
print(f"Média simulada={campanhas.mean():.6f}; variância simulada={campanhas.var():.6f}")


## 4. Como $n$ e $p$ mudam a PMF

In [ ]:
cenarios = [(10, 0.2), (10, 0.5), (30, 0.2)]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (ni, pi) in zip(axes, cenarios):
    ki = np.arange(ni + 1)
    ax.bar(ki, stats.binom.pmf(ki, ni, pi), color="#7c3aed", width=0.8)
    ax.axvline(ni*pi, color="#be123c", linestyle="--", label=f"E=np={ni*pi:g}")
    ax.set(title=f"n={ni}, p={pi}", xlabel="sucessos")
    ax.legend()
axes[0].set_ylabel("massa")
plt.tight_layout()
plt.show()


## 5. Categorical e one-hot

Uma observação escolhe exatamente uma das classes `normal`, `suspeita` e `crítica`.


In [ ]:
classes = np.array(["normal", "suspeita", "crítica"])
p_cat = np.array([0.70, 0.20, 0.10])
assert np.all(p_cat >= 0) and np.isclose(p_cat.sum(), 1)

indices = rng.choice(len(classes), size=200_000, p=p_cat)
one_hot = np.eye(len(classes), dtype=int)[indices]
freq_cat = one_hot.mean(axis=0)
assert np.allclose(freq_cat, p_cat, atol=TOL)
print(dict(zip(classes, np.round(freq_cat, 6))))


A matriz teórica de uma observação one-hot é $\operatorname{diag}(p)-pp^T$. Usamos denominador populacional (`ddof=0`) porque comparamos frequências empíricas ao mecanismo gerador.


In [ ]:
cov_cat_teorica = np.diag(p_cat) - np.outer(p_cat, p_cat)
cov_cat_empirica = np.cov(one_hot, rowvar=False, ddof=0)
assert np.allclose(cov_cat_empirica, cov_cat_teorica, atol=TOL)
assert np.allclose(cov_cat_teorica.sum(axis=1), 0)
print("Covariância teórica one-hot:")
print(np.round(cov_cat_teorica, 4))
print("Covariância empírica one-hot:")
print(np.round(cov_cat_empirica, 4))


## 6. Multinomial

Em dez ensaios, usamos $p=(0{,}5,0{,}3,0{,}2)$ e verificamos as contagens $(5,3,2)$.


In [ ]:
n_multi = 10
p_multi = np.array([0.5, 0.3, 0.2])
contagem_alvo = np.array([5, 3, 2])
p_alvo = stats.multinomial.pmf(contagem_alvo, n=n_multi, p=p_multi)
media_multi = n_multi * p_multi
cov_multi = stats.multinomial.cov(n_multi, p_multi)

assert contagem_alvo.sum() == n_multi
assert np.isclose(p_alvo, 0.08505)
assert np.allclose(media_multi, [5, 3, 2])
assert np.isclose(cov_multi[0, 1], -1.5)
assert np.allclose(cov_multi.sum(axis=1), 0)
print(f"P(N=(5,3,2)) = {p_alvo:.5f}")
print("E[N] =", media_multi)
print("Cov(N):")
print(np.round(cov_multi, 3))


In [ ]:
lotes = rng.multinomial(n_multi, p_multi, size=200_000)
media_lotes = lotes.mean(axis=0)
cov_lotes = np.cov(lotes, rowvar=False, ddof=0)
assert np.all(lotes.sum(axis=1) == n_multi)
assert np.allclose(media_lotes, media_multi, atol=0.02)
assert np.allclose(cov_lotes, cov_multi, atol=0.03)

fig, ax = plt.subplots(figsize=(7, 4))
pos = np.arange(3)
ax.bar(pos - 0.18, media_multi, 0.36, label="esperada", color="#0f766e")
ax.bar(pos + 0.18, media_lotes, 0.36, label="simulada", color="#f59e0b")
ax.set(xticks=pos, xticklabels=["classe 1", "classe 2", "classe 3"], ylabel="contagem média",
       title="Contagens multinomiais esperadas × simuladas")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()
print("Contagens médias simuladas:", np.round(media_lotes, 6))


## 7. Softmax estável e amostragem categorical

Subtrair o maior logit evita exponenciais desnecessariamente grandes sem alterar as probabilidades.


In [ ]:
def softmax_estavel(logits):
    logits = np.asarray(logits, dtype=float)
    deslocados = logits - np.max(logits)
    exp = np.exp(deslocados)
    return exp / exp.sum()

logits = np.array([1002.0, 1000.0, 999.0])
probs = softmax_estavel(logits)
assert np.all(np.isfinite(probs))
assert np.all(probs >= 0) and np.isclose(probs.sum(), 1)
tokens = np.array(["A", "B", "C"])
amostra_tokens = rng.choice(tokens, size=100_000, p=probs)
freq_tokens = np.array([(amostra_tokens == t).mean() for t in tokens])
assert np.allclose(freq_tokens, probs, atol=TOL)
print("Probabilidades softmax:", np.round(probs, 6))
print("Frequências amostradas: ", np.round(freq_tokens, 6))


## 8. Validação negativa

Uma boa implementação também rejeita parâmetros incoerentes em vez de normalizá-los silenciosamente.


In [ ]:
def validar_probabilidades(p):
    p = np.asarray(p, dtype=float)
    if p.ndim != 1 or p.size == 0:
        raise ValueError("p deve ser um vetor não vazio.")
    if np.any(~np.isfinite(p)) or np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("Probabilidades devem ser finitas, não negativas e somar 1.")
    return p

validar_probabilidades([0.7, 0.2, 0.1])
try:
    validar_probabilidades([0.7, 0.2, 0.2])
except ValueError as erro:
    print("Erro esperado:", erro)
else:
    raise AssertionError("O vetor inválido deveria ter sido rejeitado.")


## Desafio

Modele 50 impressões com CTR 0,04. Antes de executar, calcule $E[S]$, $\operatorname{Var}(S)$, $P(S=0)$ e $P(S\geq1)$. Depois confirme com `stats.binom` e uma simulação.

**Resposta:** média 2; variância 1,92; $P(S=0)=0{,}96^{50}\approx0{,}129886$; $P(S\geq1)\approx0{,}870114$.


In [ ]:
n_d, p_d = 50, 0.04
media_d = n_d * p_d
var_d = n_d * p_d * (1-p_d)
p_zero_d = stats.binom.pmf(0, n_d, p_d)
p_um_mais_d = stats.binom.sf(0, n_d, p_d)
assert np.isclose(media_d, 2.0)
assert np.isclose(var_d, 1.92)
assert np.isclose(p_zero_d + p_um_mais_d, 1)
print(f"E={media_d:.2f}; Var={var_d:.2f}; P(0)={p_zero_d:.6f}; P(≥1)={p_um_mais_d:.6f}")


## Conclusão

O laboratório conectou um ensaio a suas repetições: Bernoulli vira binomial ao somar sucessos, e categorical vira multinomial ao contar classes. As validações tornaram explícitos suporte, normalização, momentos e dependência entre contagens. Na Aula 08, o foco muda para contagens por intervalo e tempos de espera.
